# Week 10 · Day 1 (TensorFlow) — Transfer Learning: Feature Extraction

Same idea as the PyTorch notebook, in **TensorFlow / Keras**.

- Reuse a CNN pretrained on **ImageNet** instead of training from scratch.
- **Feature extraction:** freeze the pretrained backbone, train only a new head.
- Dataset: **FER2013** (7 emotions).
- Keras difference you'll feel: the training loop is one `.fit()` call.

> **Kaggle GPU:** Settings → Accelerator → GPU, then add the FER2013 dataset via Add Input.  
> Keras uses the GPU automatically — no `.to(device)` needed.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
print("TF version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

## 1. Load FER2013 with `image_dataset_from_directory`

- Keras reads class-folder datasets directly — no custom Dataset class needed.
- Each subfolder name becomes a class label.
- We load at **224×224** (ResNet's expected size).

In [ ]:
# Kaggle paths — folders containing the class subfolders
TRAIN_DIR = "/kaggle/input/fer2013/train"
TEST_DIR  = "/kaggle/input/fer2013/test"

IMG_SIZE = (224, 224)
BATCH = 64

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH,
    label_mode="int", color_mode="rgb", shuffle=True)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH,
    label_mode="int", color_mode="rgb", shuffle=False)

class_names = train_ds.class_names
n_classes = len(class_names)
print("classes:", class_names)

- FER2013 is grayscale; `color_mode="rgb"` loads it as 3 channels (ResNet needs 3).
- `prefetch` keeps the GPU fed while the CPU loads the next batch.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

## 2. Load a pretrained model (backbone only)

- `ResNet50(weights="imagenet", include_top=False)` loads the ImageNet backbone **without** its 1000-class head.
- `include_top=False` = "give me the feature extractor, not the classifier" — we'll add our own head.

In [ ]:
base_model = keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,              # drop the ImageNet head
    input_shape=(224, 224, 3))

print("backbone loaded. output feature-map shape:", base_model.output_shape)

## 3. Freeze the backbone, add a new head

- `base_model.trainable = False` → freeze all backbone weights.
- Add: **preprocessing → backbone → pooling → new Dense head** for 7 classes.
- `resnet.preprocess_input` scales pixels the way ImageNet ResNet expects (the Keras equivalent of ImageNet normalization).

In [ ]:
base_model.trainable = False    # freeze the backbone

inputs = keras.Input(shape=(224, 224, 3))
x = keras.applications.resnet50.preprocess_input(inputs)   # ImageNet preprocessing
x = base_model(x, training=False)                          # frozen backbone
x = keras.layers.GlobalAveragePooling2D()(x)               # feature map -> vector
outputs = keras.layers.Dense(n_classes, activation="softmax")(x)   # new head

model = keras.Model(inputs, outputs)
model.summary()

- In the summary, notice **most parameters are "Non-trainable"** (the frozen backbone).
- Only the small Dense head trains — that's feature extraction.

## 4. Train only the head

- `compile` picks the loss + optimizer (Keras's version of choosing them in PyTorch).
- `.fit()` runs the whole training loop for us — the four moves happen inside.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",   # integer labels
    metrics=["accuracy"])

history = model.fit(train_ds, validation_data=test_ds, epochs=5)

## 5. Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

tl_loss, tl_acc = model.evaluate(test_ds, verbose=0)
print(f"transfer-learning test accuracy: {tl_acc:.2%}")

# predictions for the confusion matrix
preds, trues = [], []
for xb, yb in test_ds:
    preds.extend(model.predict(xb, verbose=0).argmax(1))
    trues.extend(yb.numpy())

cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"Feature extraction on FER2013 — {tl_acc:.1%}")
plt.tight_layout()
plt.show()

## 6. The rematch: transfer learning vs from-scratch

- Week 9's from-scratch CNN on FER2013: **~45%** (use your class's actual number).
- Frozen pretrained backbone + a small head should beat it, with less training.
- **Why:** ImageNet features already capture the edges and shapes faces are made of.

In [ ]:
scratch_acc = 0.45   # <- your Week 9 from-scratch result

plt.bar(["From scratch\n(Week 9)", "Transfer learning\n(today)"],
        [scratch_acc * 100, tl_acc * 100], color=["gray", "green"])
plt.ylabel("test accuracy (%)")
plt.title("Same data, same effort — reuse wins")
for i, v in enumerate([scratch_acc * 100, tl_acc * 100]):
    plt.text(i, v + 1, f"{v:.0f}%", ha="center")
plt.show()

## Your turn (solo task) ✍️

- Swap ResNet50 for **MobileNetV2** and compare accuracy **and** speed.
- Starter below — same steps: load backbone → freeze → add head.
- Report: which is more accurate? which trains faster?

In [ ]:
# ===== YOUR CODE (solo task) =====
# hint:
# base = keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
# base.trainable = False
# inputs = keras.Input((224,224,3))
# x = keras.applications.mobilenet_v2.preprocess_input(inputs)
# x = base(x, training=False)
# x = keras.layers.GlobalAveragePooling2D()(x)
# outputs = keras.layers.Dense(n_classes, activation="softmax")(x)
# m = keras.Model(inputs, outputs)
# ... compile + fit as above


## PyTorch ↔ Keras (transfer learning cheat sheet)

| Step | PyTorch | Keras |
|---|---|---|
| Load backbone | `models.resnet18(weights=...)` | `ResNet50(weights="imagenet", include_top=False)` |
| Freeze | `for p in model.parameters(): p.requires_grad=False` | `base_model.trainable = False` |
| New head | replace `model.fc` with `nn.Linear` | add `GlobalAveragePooling2D` + `Dense` |
| Preprocess | `transforms.Normalize(mean, std)` | `resnet50.preprocess_input` |
| Train | you write the loop | `model.fit(...)` |
| GPU | `.to(device)` on model + batches | automatic |

- Same idea, different words — exactly like Week 8.
- We use PyTorch as our main framework; this shows you can read Keras transfer-learning code too.

## Summary

- **Feature extraction in Keras:** load a pretrained backbone with `include_top=False`, set `trainable = False`, add a pooling layer + a new `Dense` head.
- `preprocess_input` handles the ImageNet-style scaling; `image_dataset_from_directory` loads class folders directly.
- `.fit()` trains only the head (the backbone is frozen).
- Beats the from-scratch CNN on FER2013 — same lesson as the PyTorch notebook.
- Keras uses the GPU automatically; no manual device moves.

**Tomorrow:** fine-tuning (unfreeze the backbone) + a first look at object detection.